In [14]:
import pandas as pd
import os

loadind the csv file as df and filling the "home_goals" and "away_goals" columns

In [15]:
all_matches_cleaned = pd.read_csv('../data/processed/all_matches_cleaned.csv')

scores = all_matches_cleaned['result'].str.split(' - ', expand=True)

all_matches_cleaned['home_goals'] = pd.to_numeric(scores[0], errors='coerce')
all_matches_cleaned['away_goals'] = pd.to_numeric(scores[1], errors='coerce')

print(all_matches_cleaned.head())

       home_team away_team              date competition stage     season  \
0     Fiorentina    Torino  19/09/2020 18:00     Serie_A     1  2020-2021   
1  Hellas Verona      Roma  19/09/2020 20:45     Serie_A     1  2020-2021   
2          Parma    Napoli  20/09/2020 12:30     Serie_A     1  2020-2021   
3          Genoa   Crotone  20/09/2020 15:00     Serie_A     1  2020-2021   
4       Sassuolo  Cagliari  20/09/2020 18:00     Serie_A     1  2020-2021   

   is_after_ucl result  home_goals  away_goals  points  ucl_format  \
0           NaN  1 - 0           1           0     NaN         NaN   
1           NaN  0 - 0           0           0     NaN         NaN   
2           NaN  0 - 2           0           2     NaN         NaN   
3           NaN  4 - 1           4           1     NaN         NaN   
4           NaN  1 - 1           1           1     NaN         NaN   

   home_team_match_count  away_team_match_count  team_elo  opponent_elo  \
0                    NaN                 

Filling the "points" column

In [16]:
def calculate_points(row):
    if row['home_goals'] > row['away_goals']:
        return 3
    elif row['home_goals'] == row['away_goals']:
        return 1
    else:
        return 0
    
all_matches_cleaned['points'] = all_matches_cleaned.apply(calculate_points, axis=1)

print(all_matches_cleaned.head())

# I didn't care about the champions league matches because they are not relevant to the analysis in terms of points. 
# also, i didn't wanted to overcomplicate the code by adding a condition for the champions league matches, so I just left them as they are.

       home_team away_team              date competition stage     season  \
0     Fiorentina    Torino  19/09/2020 18:00     Serie_A     1  2020-2021   
1  Hellas Verona      Roma  19/09/2020 20:45     Serie_A     1  2020-2021   
2          Parma    Napoli  20/09/2020 12:30     Serie_A     1  2020-2021   
3          Genoa   Crotone  20/09/2020 15:00     Serie_A     1  2020-2021   
4       Sassuolo  Cagliari  20/09/2020 18:00     Serie_A     1  2020-2021   

   is_after_ucl result  home_goals  away_goals  points  ucl_format  \
0           NaN  1 - 0           1           0       3         NaN   
1           NaN  0 - 0           0           0       1         NaN   
2           NaN  0 - 2           0           2       0         NaN   
3           NaN  4 - 1           4           1       3         NaN   
4           NaN  1 - 1           1           1       1         NaN   

   home_team_match_count  away_team_match_count  team_elo  opponent_elo  \
0                    NaN                 

Filling the "ucl_format" column as "new" and "old"

In [17]:
new_seasons = ['2024-2025', '2025-2026']

all_matches_cleaned['ucl_format'] = pd.NA

mask = all_matches_cleaned['competition'] == 'Champions_League'

all_matches_cleaned.loc[mask, 'ucl_format'] = all_matches_cleaned.loc[mask, 'season'].apply(
    lambda x: 'new' if x in new_seasons else 'old'
)

print(all_matches_cleaned.tail())

            home_team       away_team              date       competition  \
13550       Liverpool           Paris  14/04/2026 19:00  Champions_League   
13551         Arsenal     Sporting CP  15/04/2026 19:00  Champions_League   
13552  Bayern München     Real Madrid  15/04/2026 19:00  Champions_League   
13553           Paris  Bayern München  28/04/2026 19:00  Champions_League   
13554          Atleti         Arsenal  29/04/2026 19:00  Champions_League   

           stage     season  is_after_ucl result  home_goals  away_goals  \
13550  QF Game 2  2025-2026           NaN  0 - 2           0           2   
13551  QF Game 2  2025-2026           NaN  0 - 0           0           0   
13552  QF Game 2  2025-2026           NaN  4 - 3           4           3   
13553  SF Game 1  2025-2026           NaN  5 - 4           5           4   
13554  SF Game 1  2025-2026           NaN  1 - 1           1           1   

       points ucl_format  home_team_match_count  away_team_match_count  \
13550 

Filling the "home_team_match_count" and "away_team_match_count" column. "number of matches the team had played so far in the season"

In [18]:
# make sure that that the columns are in date order. 
all_matches_cleaned = all_matches_cleaned.sort_values(by='date').reset_index(drop=True)

match_counts = {}

home_team_match_counts = []
away_team_match_counts = []

for idx, row in all_matches_cleaned.iterrows():
    season = row['season']
    home_team = row['home_team']
    away_team = row['away_team']

    # Update match counts for home and away teams
    match_counts[(home_team, season)] = match_counts.get((home_team, season), 0) + 1
    home_team_match_counts.append(match_counts[(home_team, season)])

    match_counts[(away_team, season)] = match_counts.get((away_team, season), 0) + 1
    away_team_match_counts.append(match_counts[(away_team, season)])

all_matches_cleaned['home_team_match_count'] = home_team_match_counts
all_matches_cleaned['away_team_match_count'] = away_team_match_counts

print(all_matches_cleaned.tail())

           home_team    away_team              date     competition stage  \
13550  Villarreal CF  Valencia CF  31/12/2022 15:15         La_Liga    15   
13551  Real Sociedad   CA Osasuna  31/12/2022 15:15         La_Liga    15   
13552       Brighton      Arsenal  31/12/2022 17:30  Premier_League    18   
13553         Fulham      Arsenal  31/12/2023 14:00  Premier_League    20   
13554          Spurs  Bournemouth  31/12/2023 14:00  Premier_League    20   

          season  is_after_ucl result  home_goals  away_goals  points  \
13550  2022-2023           NaN  2 - 1           2           1       3   
13551  2022-2023           NaN  2 - 0           2           0       3   
13552  2022-2023           NaN  2 - 4           2           4       0   
13553  2023-2024           NaN  2 - 1           2           1       3   
13554  2023-2024           NaN  3 - 1           3           1       3   

      ucl_format  home_team_match_count  away_team_match_count  team_elo  \
13550       <NA>      